<a href="https://colab.research.google.com/github/alilouati/data-visualization-using-python/blob/main/Chapter_4/Ch4_Dashboards_Dash_Bokeh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4: Interactive Dashboards with Dash & Bokeh (in Colab)
**Data Visualization — Ph.D. in Computing** · Dr. Ali Louati · IS Dept., PSAU

⏱️ Estimated lab time: **2 × 90 min**

### Learning objectives
By the end of this lab you should be able to:
1. Describe a dashboard's architecture: data layer → logic layer → presentation layer.
2. Run Plotly Dash apps **inside Colab** (`jupyter_mode='inline'`) — no Spyder or local server needed.
3. Write callbacks with `Input`, `Output` and `State`, including chained and cross-filtering callbacks.
4. Add KPI cards and coordinated views (Shneiderman: overview → zoom/filter → details on demand).
5. Unit-test callback logic like any other Python function.
6. Build serverless interactive dashboards with Bokeh + `CustomJS`, and quick widgets with `ipywidgets`.

### How to use this notebook
* Run cells top to bottom with **Shift + Enter** (or *Runtime → Run all*).
* Cells marked **✏️ Exercise** are for you. Cells marked **🔑 Solution** are collapsed: try first, then double-click to reveal.
* Save your own copy: *File → Save a copy in Drive*.

## 0. Setup
Dash is not pre-installed on Colab, so we install it (≈ 15 s). Since Dash 2.11 the old `jupyter-dash` package is **not** needed: `app.run(jupyter_mode="inline")` renders the app in the notebook and Dash detects Colab automatically.

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
try:
    import dash
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dash"], check=True)
    import dash
from dash import Dash, dcc, html, dash_table, Input, Output, State, callback_context
import plotly.express as px, pandas as pd, numpy as np
print("dash", dash.__version__, "| Colab:", IN_COLAB)

# One helper to run every app in this notebook. Each app gets its own port.
def show(app, port, height=560):
    app.run(jupyter_mode="inline", port=port, jupyter_height=height, debug=False)

## 1. Dashboard mental model
```
┌────────────┐   ┌────────────────────┐   ┌──────────────────────────┐
│ Data layer │ → │ Logic layer        │ → │ Presentation layer       │
│ CSV/DB/API │   │ filter, aggregate, │   │ layout: KPIs, charts,    │
│ → DataFrame│   │ model, cache       │   │ tables, controls         │
└────────────┘   └────────────────────┘   └──────────────────────────┘
                         ↑  callbacks (user events)  │
                         └───────────────────────────┘
```
**Design principle:** keep the *logic* in pure functions (DataFrame in → DataFrame/figure out). Then it is testable without a browser (Section 6).

| | **Dash (Plotly)** | **Bokeh** | **Streamlit** (Ch. 5) |
|---|---|---|---|
| Programming model | declarative layout + reactive callbacks | glyphs + widgets; JS callbacks or Bokeh server | script re-runs top → bottom |
| Best for | multi-page analytical apps, fine-grained updates | large streaming / linked plots, standalone HTML | fastest prototyping |
| Runs in Colab | ✅ inline | ✅ `output_notebook()` | ✅ via proxy/tunnel |

## 2. Minimal Dash app: Imports → Layout → Callbacks

In [ ]:
df = pd.DataFrame({"x": [1, 2, 3, 4, 2, 3, 3], "y": [10, 15, 13, 17, 11, 14, 12]})

app1 = Dash(__name__)
app1.layout = html.Div([
    html.H4("Minimal Dash app"),
    dcc.Dropdown(id="col", options=[{"label": c, "value": c} for c in df.columns], value="x", clearable=False),
    dcc.Graph(id="fig"),
])

@app1.callback(Output("fig", "figure"), Input("col", "value"))
def update_hist(col):
    return px.histogram(df, x=col, title=f"Histogram of {col}")

show(app1, 8050, height=480)

> **Running outside Colab?** In a plain `.py` file use `app.run(debug=True)` and open http://127.0.0.1:8050. In Spyder, open the browser with `webbrowser.open_new("http://127.0.0.1:8050")` from a `threading.Timer`, as on the slides.

## 3. Iris explorer: dropdown + slider + KPI cards

In [ ]:
iris = px.data.iris()
FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

def kpi(title, value_id):
    return html.Div([html.Div(title, style={"fontSize": 12, "color": "#555"}),
                     html.Div(id=value_id, style={"fontSize": 26, "fontWeight": 600})],
                    style={"padding": "8px 16px", "border": "1px solid #ddd", "borderRadius": 8, "minWidth": 120})

app2 = Dash(__name__)
app2.layout = html.Div([
    html.H4("Iris explorer"),
    html.Div([
        html.Div([html.Label("Feature"), dcc.Dropdown(FEATURES, "petal_length", id="feat", clearable=False)], style={"width": 220}),
        html.Div([html.Label("Species"), dcc.Checklist(sorted(iris.species.unique()), sorted(iris.species.unique()), id="sp", inline=True)]),
        html.Div([html.Label("Bins"), dcc.Slider(5, 50, 5, value=20, id="bins")], style={"width": 260}),
    ], style={"display": "flex", "gap": 24, "alignItems": "end"}),
    html.Div([kpi("Rows", "k_n"), kpi("Mean", "k_mean"), kpi("Std", "k_std")], style={"display": "flex", "gap": 12, "margin": "12px 0"}),
    html.Div([dcc.Graph(id="hist", style={"flex": 1}), dcc.Graph(id="box", style={"flex": 1})], style={"display": "flex"}),
], style={"fontFamily": "sans-serif"})

def filter_iris(species):                       # logic layer: pure function
    return iris[iris.species.isin(species or [])]

@app2.callback(Output("hist", "figure"), Output("box", "figure"),
               Output("k_n", "children"), Output("k_mean", "children"), Output("k_std", "children"),
               Input("feat", "value"), Input("sp", "value"), Input("bins", "value"))
def update_iris(feat, species, bins):
    d = filter_iris(species)
    h = px.histogram(d, x=feat, color="species", nbins=bins, barmode="overlay", opacity=.7, height=330)
    b = px.box(d, x="species", y=feat, color="species", points="all", height=330)
    return h, b, f"{len(d)}", f"{d[feat].mean():.2f}" if len(d) else "–", f"{d[feat].std():.2f}" if len(d) > 1 else "–"

show(app2, 8051, height=620)

## 4. Chained callbacks: choose a dataset, then its features
The second dropdown's *options* depend on the first dropdown's *value*.

In [ ]:
DATASETS = {"iris": px.data.iris(), "tips": px.data.tips(), "gapminder (2007)": px.data.gapminder().query("year == 2007")}
num_cols = lambda d: d.select_dtypes("number").columns.drop(["year", "iso_num", "species_id"], errors="ignore").tolist()
cat_cols = lambda d: d.select_dtypes(exclude="number").columns.tolist()

app3 = Dash(__name__)
app3.layout = html.Div([
    html.H4("Multi-dataset explorer"),
    html.Div([
        dcc.Dropdown(list(DATASETS), "iris", id="ds", clearable=False, style={"width": 200}),
        dcc.Dropdown(id="xcol", style={"width": 180}), dcc.Dropdown(id="ycol", style={"width": 180}),
        dcc.Dropdown(id="ccol", style={"width": 180}, placeholder="colour by"),
    ], style={"display": "flex", "gap": 10}),
    dcc.Graph(id="sc"),
])

@app3.callback(Output("xcol", "options"), Output("xcol", "value"), Output("ycol", "options"), Output("ycol", "value"),
               Output("ccol", "options"), Output("ccol", "value"), Input("ds", "value"))
def update_columns(ds):
    d = DATASETS[ds]; nums, cats = num_cols(d), cat_cols(d)
    return nums, nums[0], nums, nums[1], cats, (cats[0] if cats else None)

@app3.callback(Output("sc", "figure"), Input("ds", "value"), Input("xcol", "value"), Input("ycol", "value"), Input("ccol", "value"))
def update_scatter(ds, x, y, c):
    d = DATASETS[ds]
    if x not in d or y not in d:
        return px.scatter(title="select columns")
    return px.scatter(d, x=x, y=y, color=c if c in d else None, trendline=None, height=420, title=f"{ds}: {y} vs {x}")

show(app3, 8052, height=520)

## 5. Coordinated views: cross-filtering and `State`
*Overview first, zoom and filter, then details on demand* (Shneiderman, 1996). Use **box/lasso select** on the scatter plot: the table shows the selected countries (details on demand). The **Apply** button demonstrates `State`: the text box is read only when the button is clicked.

In [ ]:
gap07 = px.data.gapminder().query("year == 2007")

app4 = Dash(__name__)
app4.layout = html.Div([
    html.H4("Gapminder 2007 — select points to drill down"),
    html.Div([dcc.Input(id="title_in", value="Life expectancy vs GDP", style={"width": 300}),
              html.Button("Apply title", id="apply", n_clicks=0)], style={"display": "flex", "gap": 8}),
    dcc.Graph(id="overview"),
    html.Div(id="sel_info"),
    dash_table.DataTable(id="details", page_size=8, style_table={"overflowX": "auto"},
                         columns=[{"name": c, "id": c} for c in ["country", "continent", "lifeExp", "gdpPercap", "pop"]]),
])

@app4.callback(Output("overview", "figure"), Input("apply", "n_clicks"), State("title_in", "value"))
def draw_overview(_, title):
    fig = px.scatter(gap07, x="gdpPercap", y="lifeExp", size="pop", color="continent", hover_name="country",
                     log_x=True, size_max=40, height=420, title=title, custom_data=["country"])
    return fig.update_layout(dragmode="lasso")

def select_countries(selected):                 # pure logic, testable
    if not selected or not selected.get("points"):
        return gap07.iloc[0:0]
    names = [p["customdata"][0] for p in selected["points"]]
    return gap07[gap07.country.isin(names)].sort_values("lifeExp", ascending=False)

@app4.callback(Output("details", "data"), Output("sel_info", "children"), Input("overview", "selectedData"))
def show_details(selected):
    d = select_countries(selected)
    msg = "Select points with the lasso tool." if d.empty else f"{len(d)} countries · mean life expectancy {d.lifeExp.mean():.1f} years"
    return d[["country", "continent", "lifeExp", "gdpPercap", "pop"]].round(1).to_dict("records"), msg

show(app4, 8053, height=820)

## 6. Testing dashboard logic (software-engineering rigour)
Because the logic lives in pure functions, we can test it with plain `assert`s — or `pytest` in a real project. This is also how your exercises will be auto-graded.

In [ ]:
# Iris
assert len(filter_iris(["setosa"])) == 50
assert filter_iris([]).empty
h, b, n, mean, std = update_iris("petal_length", ["setosa", "virginica"], 20)
assert n == "100" and h.layout.xaxis.title.text == "petal_length"

# Chained dropdowns
opts_x, x0, opts_y, y0, cats, c0 = update_columns("tips")
assert "total_bill" in opts_x and c0 in px.data.tips().columns

# Cross-filter (simulate a lasso selection payload)
fake = {"points": [{"customdata": ["Japan"]}, {"customdata": ["Chad"]}]}
assert list(select_countries(fake).country) == ["Japan", "Chad"]
assert select_countries(None).empty
print("✅ all dashboard logic tests passed")

## 7. Bokeh: interactive dashboards without a server
`output_notebook()` renders Bokeh in Colab. With `CustomJS` callbacks the interaction runs entirely in the browser, so the result can also be saved as **one standalone HTML file** and emailed or put on a website.

In [ ]:
from bokeh.plotting import figure, show as bshow, output_notebook, output_file, save
from bokeh.models import ColumnDataSource, Select, Slider, CustomJS, HoverTool
from bokeh.layouts import column, row
from bokeh.palettes import Category10
output_notebook(hide_banner=True)

iris_b = iris.copy()
iris_b["color"] = iris_b.species.map(dict(zip(sorted(iris.species.unique()), Category10[3])))
iris_b["x"], iris_b["y"] = iris_b.sepal_length, iris_b.petal_length
src = ColumnDataSource(iris_b)

p = figure(width=520, height=380, title="Iris (Bokeh + CustomJS)", tools="pan,wheel_zoom,box_select,reset,save")
p.scatter("x", "y", source=src, color="color", size=8, alpha=.7, legend_field="species")
p.add_tools(HoverTool(tooltips=[("species", "@species"), ("x", "@x"), ("y", "@y")]))
p.xaxis.axis_label, p.yaxis.axis_label = "sepal_length", "petal_length"

sx = Select(title="X", value="sepal_length", options=FEATURES)
sy = Select(title="Y", value="petal_length", options=FEATURES)
size = Slider(title="marker size", start=3, end=20, value=8, step=1)
cb = CustomJS(args=dict(src=src, sx=sx, sy=sy, p=p), code='''
    const d = src.data;
    d['x'] = d[sx.value]; d['y'] = d[sy.value];
    p.xaxis[0].axis_label = sx.value; p.yaxis[0].axis_label = sy.value;
    src.change.emit();
''')
sx.js_on_change("value", cb); sy.js_on_change("value", cb)
size.js_on_change("value", CustomJS(args=dict(r=p.renderers[0]), code="r.glyph.size = cb_obj.value;"))
layout = row(column(sx, sy, size), p)
bshow(layout)

output_file("iris_bokeh_dashboard.html", title="Iris Bokeh dashboard"); save(layout)
print("Saved standalone file: iris_bokeh_dashboard.html")

## 8. The lightest option: `ipywidgets`
For quick exploration inside a notebook, `interact` turns function arguments into widgets.

In [ ]:
import ipywidgets as w
import matplotlib.pyplot as plt
if IN_COLAB:
    from google.colab import output; output.enable_custom_widget_manager()

@w.interact(feature=FEATURES, bins=w.IntSlider(value=20, min=5, max=60, step=5))
def quick_hist(feature="petal_length", bins=20):
    plt.figure(figsize=(6, 3))
    for sp, d in iris.groupby("species"):
        plt.hist(d[feature], bins=bins, alpha=.6, label=sp)
    plt.legend(); plt.xlabel(feature); plt.show()

## 9. ✏️ Exercises
**E1 (in class, 20 min).** Pick `tips`, `flights` (`px.data` has `tips`, `gapminder`, `wind`, `stocks`, `election`) or your own data. Build **1 dropdown + 1 graph** in Dash. Then add **one KPI card**.

**E2 (in class, 30 min).** Add a `dcc.RangeSlider` for a numeric column to the iris app and a **second** graph that cross-filters from the first (`selectedData`). Write two `assert` tests for your new logic.

**E3 (take-home).** Re-implement your E2 dashboard in **Bokeh** as a standalone HTML file. Compare the two on: lines of code, latency, deployability, and testability (≤ 1 page).

**E4 (research).** Pick a dashboard from your PhD field or workplace. Evaluate it with Few's *Information Dashboard Design* guidelines and with a 5-user think-aloud study plan (tasks, metrics, threats to validity).

In [ ]:
# ✏️ E1 — start from this skeleton
tips = px.data.tips()
app_ex = Dash(__name__)
app_ex.layout = html.Div([
    dcc.Dropdown(["total_bill", "tip", "size"], "tip", id="ex_col", clearable=False),
    dcc.Graph(id="ex_fig"),
    # TODO: add a KPI card
])

@app_ex.callback(Output("ex_fig", "figure"), Input("ex_col", "value"))
def ex_update(col):
    return px.histogram(tips, x=col, color="time")   # TODO: improve

show(app_ex, 8060, height=480)

In [ ]:
#@title 🔑 Solution E2 (range slider + linked second graph) (try first, then double-click to reveal) { display-mode: "form" }
app_sol = Dash(__name__)
app_sol.layout = html.Div([
    dcc.RangeSlider(4, 8, 0.1, value=[4, 8], id="sl_range", marks={i: str(i) for i in range(4, 9)}),
    html.Div([dcc.Graph(id="g_main", style={"flex": 1}), dcc.Graph(id="g_detail", style={"flex": 1})], style={"display": "flex"}),
])

def range_filter(lo, hi):
    return iris[iris.sepal_length.between(lo, hi)]

@app_sol.callback(Output("g_main", "figure"), Input("sl_range", "value"))
def main_fig(rng_):
    d = range_filter(*rng_)
    return px.scatter(d, x="sepal_length", y="petal_length", color="species", custom_data=["species"], height=360).update_layout(dragmode="select")

@app_sol.callback(Output("g_detail", "figure"), Input("g_main", "selectedData"), Input("sl_range", "value"))
def detail_fig(sel, rng_):
    d = range_filter(*rng_)
    if sel and sel.get("points"):
        d = d[d.sepal_length.isin([p["x"] for p in sel["points"]]) & d.petal_length.isin([p["y"] for p in sel["points"]])]
    return px.histogram(d, x="petal_width", color="species", height=360, title=f"{len(d)} selected")

assert len(range_filter(4, 8)) == 150 and range_filter(7.5, 8).species.eq("virginica").all()
show(app_sol, 8061, height=480)

## References
* Shneiderman, B. (1996). *The Eyes Have It: A Task by Data Type Taxonomy for Information Visualizations*. IEEE VL.
* Few, S. (2013). *Information Dashboard Design*, 2nd ed. Analytics Press.
* Sarikaya, A. et al. (2019). *What Do We Talk About When We Talk About Dashboards?* IEEE TVCG.
* Roberts, J. C. (2007). *State of the Art: Coordinated & Multiple Views in Exploratory Visualization*. CMV.
* Plotly Dash docs — *Dash in Jupyter environments*: https://dash.plotly.com/dash-in-jupyter